# Calibrated Popularity and history-aware calibration

This notebook adds the two re-ranking methods the proposal describes on top of the ALS baseline: **fixed Calibrated Popularity (CP)** and **history-aware CP**. It then evaluates every setting with UPD, Precision@10, Recall@10 and NDCG@10, broken down by overall, history bucket and user segment.

**Inputs:** the exported `top-100-candidates` and `top-10-baseline` Parquet tables from `als-baseline.ipynb`, plus `data/ml-1m/ratings.dat`. The training/test split is rebuilt with the same seed and hash as the baseline, then verified against the exported candidates.

**Method (proposal Section 2.2):** for each user, build the top-10 greedily. At each step add the candidate that maximizes `(1 - lambda_u) * Rel(L) - lambda_u * JS(p(u), q(L))`. CP uses `lambda_u = lambda`. History-aware CP uses `lambda_u = lambda * min(1, n_u / N0)`, where `n_u` is the user's number of training ratings.

**Design choice to confirm with the team:** greedy selection decides *which* 10 movies are shown. The final list is then ordered by the ALS rank, so calibration changes list membership but not the relative order of what remains. At `lambda = 0` this reproduces the baseline exactly, and the notebook asserts that. A tiny positive weight is not assumed to be equivalent because it can break tied relevance scores differently.

Run the cells in order with the Scala 2.12.18 / Java 17 kernel. Everything after the Spark steps runs on the driver; the data is small (604,000 candidate rows).

In [4]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

import $ivy.$
import $ivy.$
import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

In [5]:
val spark = SparkSession.builder()
  .appName("CS 7265 - Calibrated Popularity")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

assert(spark.version == "3.5.9")
println(s"Spark ${spark.version}, Scala ${scala.util.Properties.versionNumberString}")

Spark 3.5.9, Scala 2.12.18


spark: SparkSession = org.apache.spark.sql.SparkSession@3b9fa144
import spark.implicits._

## 1. Configuration

Set `baselineRunId` to the timestamped folder under `outputs/als-baseline/` for the recommendations to calibrate, `profileRunId` to the matching run under `outputs/popularity-profiles/`, and `splitSeed` to the seed used by that ALS run. Run `popularity-profiles.ipynb` with the same split first. The next section requires an exact match between the shared movie labels and profiles and this notebook's independently rebuilt values. The grids follow the proposal; `lambda = 0.0` is added as the unadjusted reference point.

In [ ]:
val baselineRunId = "2026-11-12T07-10-03.780907Z"   // e.g. 2026-11-02T18-04-11.123456Z
val profileRunId = ""   // timestamped run under outputs/popularity-profiles/
val splitSeed = 42L
val trainingFraction = 0.8
val listSize = 10
val candidatePoolSize = 100

val lambdas = Seq(0.1, 0.3, 0.5, 0.7, 0.9)
val n0Values = Seq(50, 100, 200)

val shortCutoff = 50     // short:  n_u < 50
val longCutoff = 200     // medium: 50 <= n_u <= 200, long: n_u > 200

// Optional historical regression check; applies only to this baseline run.
val historicalRegressionBaselineRunId = "2026-11-12T07-10-03.780907Z"
val checkHistoricalBaselineRegression = false
val historicalPrecision = 0.0567
val historicalRecall = 0.0391
val historicalNdcg = 0.0513

val ratingsPath = Seq("data/ml-1m/ratings.dat", "../data/ml-1m/ratings.dat")
  .find(p => new java.io.File(p).isFile)
  .getOrElse(throw new IllegalArgumentException("Run from the repository root or notebooks folder with data/ml-1m/ present."))
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile.getParentFile.getParentFile.getParentFile
val baselineRunDir = new java.io.File(repositoryRoot, s"outputs/als-baseline/$baselineRunId")
require(baselineRunDir.isDirectory, s"Baseline run folder not found: ${baselineRunDir.getPath}")
require(profileRunId.nonEmpty, "Set profileRunId to the matching outputs/popularity-profiles run.")
val profileRunDir = new java.io.File(repositoryRoot, s"outputs/popularity-profiles/$profileRunId")
require(profileRunDir.isDirectory, s"Popularity profile run folder not found: ${profileRunDir.getPath}")

val calibrationRunId = java.time.Instant.now().toString.replace(":", "-")
val calibrationDir = new java.io.File(repositoryRoot, s"outputs/calibration/$calibrationRunId")
println(s"Baseline run:    ${baselineRunDir.getPath}")
println(s"Calibration run: ${calibrationDir.getPath}")

## 2. Rebuild the training/test split

This repeats the baseline's per-user split: `xxhash64(splitSeed, userId, movieId)` ordering, `floor(0.8 * n)` training rows per user. The hash depends on the column types, so `userId` and `movieId` are read as `Int` exactly as in the baseline. Section 3 checks the result against the exported candidates.

In [7]:
val ratingSchema = StructType(Seq(
  StructField("userId", IntegerType, false),
  StructField("movieId", IntegerType, false),
  StructField("rating", FloatType, false),
  StructField("timestamp", LongType, false)
))

val ratingsDF = spark.read
  .option("sep", "::")
  .schema(ratingSchema)
  .csv(ratingsPath)

val ratingCount = ratingsDF.count()
assert(ratingCount == 1000209L, s"Expected 1,000,209 ratings, found $ratingCount")

val splitRanked = ratingsDF
  .withColumn("userRatingCount", count("*").over(Window.partitionBy("userId")))
  .withColumn("shuffleKey", xxhash64(lit(splitSeed), col("userId"), col("movieId")))
  .withColumn("ratingPosition", row_number().over(
    Window.partitionBy("userId").orderBy(col("shuffleKey"), col("movieId"))))
  .withColumn("trainingCount", floor(col("userRatingCount") * trainingFraction))

val trainingRatings = splitRanked.filter(col("ratingPosition") <= col("trainingCount"))
  .select("userId", "movieId", "rating", "timestamp").cache()
val testRatings = splitRanked.filter(col("ratingPosition") > col("trainingCount"))
  .select("userId", "movieId", "rating", "timestamp").cache()

val trainingRatingCount = trainingRatings.count()
val testRatingCount = testRatings.count()
assert(trainingRatingCount + testRatingCount == ratingCount)
println(s"Training: $trainingRatingCount ratings, test: $testRatingCount ratings")

Training: 797758 ratings, test: 202451 ratings


ratingSchema: StructType = StructType(
  StructField("userId", IntegerType, false, {}),
  StructField("movieId", IntegerType, false, {}),
  StructField("rating", FloatType, false, {}),
  StructField("timestamp", LongType, false, {})
)
ratingsDF: DataFrame = [userId: int, movieId: int ... 2 more fields]
ratingCount: Long = 1000209L
splitRanked: DataFrame = [userId: int, movieId: int ... 6 more fields]
trainingRatings: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
testRatings: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
trainingRatingCount: Long = 797758L
testRatingCount: Long = 202451L

## 3. Load the baseline exports and verify they match this split

The exported candidates must contain no training pair, and the top-10 baseline must be exactly the first ten candidates. A mismatch means the split rebuilt here differs from the one used for ALS.

In [8]:
val candidates = spark.read.parquet(new java.io.File(baselineRunDir, "top-100-candidates").getPath)
  .select("userId", "movieId", "recommendationRank", "relevanceScore")
val baselineTop10 = spark.read.parquet(new java.io.File(baselineRunDir, "top-10-baseline").getPath)
  .select("userId", "movieId", "recommendationRank", "relevanceScore")

val trainingUsers = trainingRatings.select("userId").distinct()
val nUsers = trainingUsers.count()

val candidatesInTraining = candidates.join(
  trainingRatings.select("userId", "movieId"), Seq("userId", "movieId"), "inner").count()
val candidateRows = candidates.count()
val baselineFromCandidates = candidates.filter(col("recommendationRank") <= listSize)
val baselineMismatch = baselineTop10.exceptAll(baselineFromCandidates).count() +
  baselineFromCandidates.exceptAll(baselineTop10).count()

println(s"Users: $nUsers, candidate rows: $candidateRows")
println(s"Candidate pairs found in rebuilt training split: $candidatesInTraining")
println(s"Baseline vs first-10 candidate mismatches: $baselineMismatch")
assert(candidatesInTraining == 0L, "Split mismatch: candidates overlap the rebuilt training set.")
assert(candidateRows == nUsers * candidatePoolSize, "Every user must have 100 candidates.")
assert(baselineMismatch == 0L, "Exported baseline must equal the first 10 candidates.")

Users: 6040, candidate rows: 604000
Candidate pairs found in rebuilt training split: 0
Baseline vs first-10 candidate mismatches: 0


candidates: DataFrame = [userId: int, movieId: int ... 2 more fields]
baselineTop10: DataFrame = [userId: int, movieId: int ... 2 more fields]
trainingUsers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
nUsers: Long = 6040L
candidatesInTraining: Long = 0L
candidateRows: Long = 604000L
baselineFromCandidates: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
baselineMismatch: Long = 0L

## 4. Build the inputs the evaluators need

These are the tables `evaluateUPD` documents but the repo does not yet contain. Everything is derived from **training data only**.

- **Movie popularity groups:** the shared preparation export is canonical. A movie is Head (`H`) while the cumulative share before it is under 20%, Tail (`T`) once its cumulative share including the movie crosses 80%, otherwise Mid (`M`).
- **User profiles `p(u)`:** each user's share of training rating *values* in H, M, T, following the proposal.
- **Segments:** users split into three equal groups by Head share (highest third is `blockbuster-focused`, lowest third is `niche-focused`).
- **History buckets:** short `n_u < 50`, medium `50-200`, long `> 200`, using training rating counts.

In [ ]:
val headShare = 0.20
val tailShare = 0.20
val totalTrainingRatings = trainingRatingCount.toDouble
val movieCounts = trainingRatings.groupBy("movieId").agg(count("*").as("ratingCount"))

val popularityOrder = Window
  .orderBy(col("ratingCount").desc, col("movieId").asc)
  .rowsBetween(Window.unboundedPreceding, Window.currentRow)

val generatedMovieGroups = movieCounts
  .withColumn("cumulativeShare",
    sum(col("ratingCount")).over(popularityOrder) / lit(totalTrainingRatings))
  .withColumn("shareBefore",
    col("cumulativeShare") - col("ratingCount") / lit(totalTrainingRatings))
  .withColumn("popularityGroup",
    when(col("shareBefore") < headShare, "H")
      .when(col("cumulativeShare") > 1.0 - tailShare, "T")
      .otherwise("M"))
  .select("movieId", "ratingCount", "cumulativeShare", "popularityGroup")
  .cache()

val preparedMovieGroups = spark.read.parquet(
  new java.io.File(profileRunDir, "movie-popularity-groups").getPath)
  .select("movieId", "popularityGroup")
val profileConfig = new String(java.nio.file.Files.readAllBytes(
  new java.io.File(profileRunDir, "split-config.txt").toPath), "UTF-8")
require(profileConfig.contains(s"splitSeed=$splitSeed\n") &&
  profileConfig.contains(s"trainingFraction=$trainingFraction\n") &&
  profileConfig.contains(s"trainingRatings=$trainingRatingCount\n"),
  "Popularity profile export was generated from a different training split.")
val generatedMovieLabels = generatedMovieGroups.select("movieId", "popularityGroup")
val movieLabelMismatch = preparedMovieGroups.exceptAll(generatedMovieLabels).count() +
  generatedMovieLabels.exceptAll(preparedMovieGroups).count()
assert(movieLabelMismatch == 0L,
  s"Prepared and regenerated movie labels differ in $movieLabelMismatch rows.")
val movieGroupsOut = preparedMovieGroups.cache()
val movieGroups = generatedMovieGroups

println(s"Exact movie-label comparison passed for ${movieGroupsOut.count()} movies.")
movieGroups.groupBy("popularityGroup")
  .agg(count("*").as("movies"), (sum("ratingCount") / lit(totalTrainingRatings)).as("ratingShare"))
  .orderBy("popularityGroup").show()

In [ ]:
val ratingValue = col("rating").cast("double")
val generatedProfiles = trainingRatings.join(generatedMovieLabels, "movieId")
  .groupBy("userId")
  .agg(
    sum(ratingValue).as("total"),
    sum(when(col("popularityGroup") === "H", ratingValue).otherwise(0.0)).as("h"),
    sum(when(col("popularityGroup") === "M", ratingValue).otherwise(0.0)).as("m"),
    sum(when(col("popularityGroup") === "T", ratingValue).otherwise(0.0)).as("t"))
  .select(col("userId"),
    (col("h") / col("total")).as("head"),
    (col("m") / col("total")).as("mid"),
    (col("t") / col("total")).as("tail"))
  .cache()

val preparedProfiles = spark.read.parquet(
  new java.io.File(profileRunDir, "user-popularity-profiles").getPath)
  .select("userId", "head", "mid", "tail")
val generatedProfileValues = generatedProfiles.select("userId", "head", "mid", "tail")
val profileMismatch = preparedProfiles.exceptAll(generatedProfileValues).count() +
  generatedProfileValues.exceptAll(preparedProfiles).count()
assert(profileMismatch == 0L,
  s"Prepared and regenerated user profiles differ in $profileMismatch rows.")
val profiles = preparedProfiles.cache()
println(s"Exact profile comparison passed for ${profiles.count()} users.")

val segmentTile = ntile(3).over(Window.orderBy(col("head").desc, col("userId").asc))
val segments = profiles.withColumn("tile", segmentTile)
  .select(col("userId"),
    when(col("tile") === 1, "blockbuster-focused")
      .when(col("tile") === 2, "diverse")
      .otherwise("niche-focused").as("segment"))
  .cache()

val userHistory = trainingRatings.groupBy("userId").agg(count("*").as("nTrain"))
  .withColumn("historyBucket",
    when(col("nTrain") < shortCutoff, "short")
      .when(col("nTrain") <= longCutoff, "medium")
      .otherwise("long"))
  .cache()

assert(profiles.count() == nUsers && segments.count() == nUsers && userHistory.count() == nUsers)
println("Segments and history buckets:")
segments.groupBy("segment").count().orderBy("segment").show()
userHistory.groupBy("historyBucket").agg(count("*").as("users"), min("nTrain"), max("nTrain"))
  .orderBy("historyBucket").show()

## 5. Calibration engine

Everything here runs on the driver with plain Scala arrays, so nothing is serialized to Spark workers. `Calib.js` is base-2 Jensen-Shannon divergence, not square-rooted, so it matches the UPD notebook and ranges over 0-1.

In [ ]:
case class Cand(movieId: Int, rank: Int, rel: Double, g: Int)   // g: 0 = H, 1 = M, 2 = T

object Calib {
  private val ln2 = math.log(2.0)

  def js(p: Array[Double], q: Array[Double]): Double = {
    var s = 0.0
    var i = 0
    while (i < 3) {
      val m = (p(i) + q(i)) / 2.0
      if (p(i) > 0.0) s += 0.5 * p(i) * math.log(p(i) / m) / ln2
      if (q(i) > 0.0) s += 0.5 * q(i) * math.log(q(i) / m) / ln2
      i += 1
    }
    s
  }

  def mix(list: Array[Cand]): Array[Double] = {
    val c = Array(0.0, 0.0, 0.0)
    list.foreach(x => c(x.g) += 1.0)
    c.map(_ / list.length)
  }

  // cands must be sorted by ALS rank. Ties go to the better-ranked candidate.
  def rerank(cands: Array[Cand], p: Array[Double], lambda: Double, k: Int): Array[Cand] = {
    val used = new Array[Boolean](cands.length)
    val chosen = new Array[Cand](k)
    val counts = Array(0.0, 0.0, 0.0)
    val q = new Array[Double](3)
    var relSum = 0.0
    var t = 0
    while (t < k) {
      val size = (t + 1).toDouble
      var best = -1
      var bestObj = Double.NegativeInfinity
      var i = 0
      while (i < cands.length) {
        if (!used(i)) {
          val g = cands(i).g
          counts(g) += 1.0
          q(0) = counts(0) / size; q(1) = counts(1) / size; q(2) = counts(2) / size
          counts(g) -= 1.0
          val obj = (1.0 - lambda) * (relSum + cands(i).rel) - lambda * js(p, q)
          if (obj > bestObj) { bestObj = obj; best = i }
        }
        i += 1
      }
      used(best) = true
      chosen(t) = cands(best)
      relSum += cands(best).rel
      counts(cands(best).g) += 1.0
      t += 1
    }
    chosen.sortBy(_.rank)    // present in ALS order; calibration only changes membership
  }
}

// JS sanity checks
val pHead = Array(1.0, 0.0, 0.0); val pTail = Array(0.0, 0.0, 1.0); val pMix = Array(0.2, 0.5, 0.3)
assert(math.abs(Calib.js(pHead, pTail) - 1.0) < 1e-12)
assert(math.abs(Calib.js(pMix, pMix)) < 1e-12)
assert(math.abs(Calib.js(pMix, pHead) - Calib.js(pHead, pMix)) < 1e-12)

val tiedCandidates = Array(Cand(1, 1, 1.0, 0), Cand(2, 2, 1.0, 2))
val tiedBaseline = Calib.rerank(tiedCandidates, pTail, 0.0, 1)
val tiedPositiveWeight = Calib.rerank(tiedCandidates, pTail, 1e-6, 1)
assert(tiedBaseline.map(_.movieId).toSeq == Seq(1), "At lambda=0, tied relevance must keep the ALS-ranked candidate.")
assert(tiedPositiveWeight.map(_.movieId).toSeq == Seq(2),
  "A tiny positive calibration weight may break a relevance tie in favor of the user's popularity profile.")
println("JS and tied-score sanity checks passed.")

In [ ]:
val groupIndex = Map("H" -> 0, "M" -> 1, "T" -> 2)

val candsByUser: Map[Int, Array[Cand]] = candidates.join(movieGroupsOut, "movieId")
  .select("userId", "movieId", "recommendationRank", "relevanceScore", "popularityGroup")
  .collect()
  .map(r => r.getInt(0) -> Cand(r.getInt(1), r.getInt(2), r.getDouble(3), groupIndex(r.getString(4))))
  .groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).sortBy(_.rank) }

val profileByUser: Map[Int, Array[Double]] = profiles.collect()
  .map(r => r.getInt(0) -> Array(r.getDouble(1), r.getDouble(2), r.getDouble(3))).toMap
val nTrainByUser: Map[Int, Long] = userHistory.select("userId", "nTrain").collect()
  .map(r => r.getInt(0) -> r.getLong(1)).toMap
val segmentByUser: Map[Int, String] = segments.collect().map(r => r.getInt(0) -> r.getString(1)).toMap
val bucketByUser: Map[Int, String] = userHistory.select("userId", "historyBucket").collect()
  .map(r => r.getInt(0) -> r.getString(1)).toMap

val likedByUser: Map[Int, Set[Int]] = testRatings.filter(col("rating") >= 4.0)
  .select("userId", "movieId").collect()
  .map(r => r.getInt(0) -> r.getInt(1)).groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).toSet }

assert(candsByUser.size == nUsers.toInt)
assert(candsByUser.values.forall(_.length == candidatePoolSize), "Every candidate movie must have a popularity label.")
println(s"Users with candidates: ${candsByUser.size}; users with a liked test movie: ${likedByUser.size}")

## 6. Settings and per-user evaluation

A `Setting` is one method with its parameters. `userLambda` applies the history-aware weight. UPD per user is the Jensen-Shannon divergence between `p(u)` and the final list's mix. Relevance metrics use test movies rated 4 or higher, and only users with at least one such movie are scored, matching `accuracy-evaluation.ipynb`.

In [ ]:
case class Setting(method: String, lambda: Double, n0: Option[Int]) {
  def name: String = (method, n0) match {
    case ("baseline", _) => "baseline"
    case ("cp", _)       => f"cp_l$lambda%.1f"
    case (_, Some(n))    => f"ha_n${n}_l$lambda%.1f"
    case _               => method
  }
  def userLambda(n: Long): Double = n0 match {
    case Some(n0v) => lambda * math.min(1.0, n.toDouble / n0v)
    case None      => lambda
  }
}

val historyWeightCheck = Setting("ha", 0.9, Some(100)).userLambda(50)
assert(math.abs(historyWeightCheck - 0.45) < 1e-12,
  s"History-aware weight should be lambda * min(1, n / N0); found $historyWeightCheck.")
assert(math.abs(Setting("ha", 0.9, Some(100)).userLambda(200) - 0.9) < 1e-12,
  "History-aware weight must cap at lambda when n >= N0.")
assert(Setting("ha", 0.9, Some(100)).userLambda(0) == 0.0,
  "History-aware weight must be zero for an empty history.")
println("History-aware weight formula checks passed.")

val settings: Seq[Setting] =
  Seq(Setting("baseline", 0.0, None)) ++
  lambdas.map(l => Setting("cp", l, None)) ++
  (for (n0 <- n0Values; l <- lambdas) yield Setting("ha", l, Some(n0)))

def runSetting(s: Setting): Map[Int, Array[Cand]] =
  candsByUser.map { case (u, cands) =>
    u -> Calib.rerank(cands, profileByUser(u), s.userLambda(nTrainByUser(u)), listSize)
  }

case class UserRec(userId: Int, segment: String, bucket: String, mismatch: Double,
                   scored: Boolean, precision: Double, recall: Double, ndcg: Double)

def evaluate(lists: Map[Int, Array[Cand]]): Seq[UserRec] =
  lists.toSeq.map { case (u, list) =>
    val mismatch = Calib.js(profileByUser(u), Calib.mix(list))
    likedByUser.get(u) match {
      case Some(liked) =>
        val hits = list.map(c => liked.contains(c.movieId))
        val nHits = hits.count(identity).toDouble
        val dcg = hits.zipWithIndex.collect { case (true, i) => 1.0 / (math.log(i + 2.0) / math.log(2.0)) }.sum
        val idcg = (0 until math.min(liked.size, listSize))
          .map(i => 1.0 / (math.log(i + 2.0) / math.log(2.0))).sum
        UserRec(u, segmentByUser(u), bucketByUser(u), mismatch, true,
          nHits / listSize, nHits / liked.size, dcg / idcg)
      case None =>
        UserRec(u, segmentByUser(u), bucketByUser(u), mismatch, false, 0.0, 0.0, 0.0)
    }
  }

## 7. Verify the engine before the sweep

1. At `lambda = 0` the engine must return the exported baseline, in the same order.
2. Check the history-aware weight formula directly; a tiny positive weight is not required to reproduce the baseline because it can break relevance ties differently.
3. Evaluate the selected exported baseline against this split's held-out ratings, then verify the calibration engine reproduces those metrics. An optional historical regression check applies only to its matching baseline run.

In [ ]:
val baselineLists = runSetting(settings.head)
val exportedBaselineByUser: Map[Int, Seq[Int]] = baselineTop10.collect()
  .map(r => (r.getInt(0), r.getInt(2), r.getInt(1))).groupBy(_._1)
  .map { case (u, xs) => u -> xs.sortBy(_._2).map(_._3).toSeq }

val baselineDiffs = baselineLists.count { case (u, l) => l.map(_.movieId).toSeq != exportedBaselineByUser(u) }
assert(baselineDiffs == 0, s"$baselineDiffs users differ between lambda=0 and the exported baseline.")

val exportedBaselineLists: Map[Int, Array[Cand]] = baselineTop10.join(movieGroupsOut, "movieId")
  .select("userId", "movieId", "recommendationRank", "relevanceScore", "popularityGroup")
  .collect()
  .map(r => r.getInt(0) -> Cand(r.getInt(1), r.getInt(2), r.getDouble(3), groupIndex(r.getString(4))))
  .groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).sortBy(_.rank) }
assert(exportedBaselineLists.size == nUsers.toInt &&
  exportedBaselineLists.values.forall(_.length == listSize),
  "The selected exported baseline must contain exactly one complete top-10 list per training user.")

def avgOf(xs: Seq[Double]) = if (xs.isEmpty) Double.NaN else xs.sum / xs.size
val exportedBaselineRecs = evaluate(exportedBaselineLists).filter(_.scored)
val calibrationBaselineRecs = evaluate(baselineLists).filter(_.scored)
val exportedPrecision = avgOf(exportedBaselineRecs.map(_.precision))
val exportedRecall = avgOf(exportedBaselineRecs.map(_.recall))
val exportedNdcg = avgOf(exportedBaselineRecs.map(_.ndcg))
val bP = avgOf(calibrationBaselineRecs.map(_.precision))
val bR = avgOf(calibrationBaselineRecs.map(_.recall))
val bN = avgOf(calibrationBaselineRecs.map(_.ndcg))
assert(calibrationBaselineRecs.size == exportedBaselineRecs.size &&
  math.abs(bP - exportedPrecision) < 1e-12 &&
  math.abs(bR - exportedRecall) < 1e-12 &&
  math.abs(bN - exportedNdcg) < 1e-12,
  "Lambda=0 accuracy must match the selected exported baseline evaluated on this split's held-out ratings.")
println(f"Selected exported baseline: users=${exportedBaselineRecs.size}  P@10=$exportedPrecision%.4f  R@10=$exportedRecall%.4f  NDCG@10=$exportedNdcg%.4f")
println(f"Calibration lambda=0:       users=${calibrationBaselineRecs.size}  P@10=$bP%.4f  R@10=$bR%.4f  NDCG@10=$bN%.4f")

if (checkHistoricalBaselineRegression) {
  require(baselineRunId == historicalRegressionBaselineRunId,
    s"Historical accuracy applies only to baseline $historicalRegressionBaselineRunId, not $baselineRunId.")
  assert(math.abs(exportedPrecision - historicalPrecision) < 5e-4 &&
    math.abs(exportedRecall - historicalRecall) < 5e-4 &&
    math.abs(exportedNdcg - historicalNdcg) < 5e-4,
    "Selected baseline differs from its historical regression metrics.")
  println("Historical baseline regression check passed.")
}
println("Engine checks passed.")

## 8. Run every setting

Each setting is re-ranked for all users, evaluated, and its top-10 is exported to Parquet in the schema `evaluateUPD` accepts. Results are aggregated for the whole cohort, each history bucket and each segment. `overallUPD` follows the README definition (equal-weight mean of the three segment means); the group rows use the plain mean mismatch within the group.

In [ ]:
case class ResultRow(setting: String, method: String, lambda: Double, n0: Int,
                     groupType: String, group: String, users: Int, scoredUsers: Int,
                     upd: Double, precision: Double, recall: Double, ndcg: Double)

def summarize(s: Setting, groupType: String, group: String, recs: Seq[UserRec]): ResultRow = {
  val scored = recs.filter(_.scored)
  ResultRow(s.name, s.method, s.lambda, s.n0.getOrElse(0), groupType, group, recs.size, scored.size,
    avgOf(recs.map(_.mismatch)), avgOf(scored.map(_.precision)), avgOf(scored.map(_.recall)), avgOf(scored.map(_.ndcg)))
}

def officialOverallUPD(recs: Seq[UserRec]): Double =
  avgOf(recs.groupBy(_.segment).values.map(g => avgOf(g.map(_.mismatch))).toSeq)

calibrationDir.mkdirs()
val allRows = scala.collection.mutable.ArrayBuffer[ResultRow]()

settings.foreach { s =>
  val lists = runSetting(s)
  assert(lists.values.forall(l => l.length == listSize && l.map(_.movieId).distinct.length == listSize))
  val recs = evaluate(lists)

  allRows += summarize(s, "all", "all", recs).copy(upd = officialOverallUPD(recs))
  Seq("short", "medium", "long").foreach(b => allRows += summarize(s, "history", b, recs.filter(_.bucket == b)))
  Seq("blockbuster-focused", "diverse", "niche-focused").foreach(g =>
    allRows += summarize(s, "segment", g, recs.filter(_.segment == g)))

  lists.toSeq.flatMap { case (u, l) =>
    l.zipWithIndex.map { case (c, i) => (u, c.movieId, i + 1, c.rel) }
  }.toDF("userId", "movieId", "recommendationRank", "relevanceScore")
    .coalesce(1).write.mode("errorifexists")
    .parquet(new java.io.File(calibrationDir, s"top-10-${s.name}").getPath)

  val a = allRows.find(r => r.setting == s.name && r.groupType == "all").get
  println(f"${s.name}%-16s UPD=${a.upd}%.4f  P@10=${a.precision}%.4f  R@10=${a.recall}%.4f  NDCG@10=${a.ndcg}%.4f")
}

## 9. Save results and the evaluator inputs

`results.csv` feeds `analysis/analyze_results.py`. The Parquet inputs use the exact schemas in the UPD notebook's Section 9, so `evaluateUPD` can be run on any exported `top-10-*` folder as an independent check of the numbers above. `manifest.txt` records the baseline and popularity-profile runs, split settings, label-boundary convention, and calibration parameters for reproducibility.

In [ ]:
val manifestText = Seq(
  s"calibrationRunId=$calibrationRunId",
  s"baselineRunId=$baselineRunId",
  s"profileRunId=$profileRunId",
  s"splitSeed=$splitSeed",
  s"trainingFraction=$trainingFraction",
  s"trainingRatings=$trainingRatingCount",
  s"testRatings=$testRatingCount",
  s"trainingUsers=$nUsers",
  s"splitMethod=per-user hash ordering, floor(trainingFraction * userRatingCount)",
  s"headShare=$headShare",
  s"headBoundary=shareBefore < headShare",
  s"tailShare=$tailShare",
  s"tailBoundary=cumulativeShare > 1.0 - tailShare (includes crossing movie)",
  "movieSort=ratingCount descending, movieId ascending",
  "calibrationMethods=baseline,CP,HA",
  "calibrationObjective=(1 - lambda_u) * Rel(L) - lambda_u * JS(p(u), q(L))",
  s"listSize=$listSize",
  s"candidatePoolSize=$candidatePoolSize",
  s"cpLambdas=${lambdas.mkString(",")}",
  s"haLambdas=${lambdas.mkString(",")}",
  s"haN0Values=${n0Values.mkString(",")}",
  s"shortHistoryCutoff=$shortCutoff",
  s"longHistoryCutoff=$longCutoff",
  s"relevanceThreshold=4.0",
  s"historicalRegressionCheckEnabled=$checkHistoricalBaselineRegression",
  s"historicalRegressionBaselineRunId=$historicalRegressionBaselineRunId",
  s"historicalPrecision=$historicalPrecision",
  s"historicalRecall=$historicalRecall",
  s"historicalNdcg=$historicalNdcg"
).mkString("\n") + "\n"
val manifestFile = new java.io.File(calibrationDir, "manifest.txt")
java.nio.file.Files.write(manifestFile.toPath, manifestText.getBytes("UTF-8"))
println(s"Run manifest: ${manifestFile.getPath}")

val csvFile = new java.io.File(calibrationDir, "results.csv")
val out = new java.io.PrintWriter(csvFile)
out.println("setting,method,lambda,n0,groupType,group,users,scoredUsers,upd,precision,recall,ndcg")
allRows.foreach(r => out.println(Seq(r.setting, r.method, r.lambda, r.n0, r.groupType, r.group,
  r.users, r.scoredUsers, r.upd, r.precision, r.recall, r.ndcg).mkString(",")))
out.close()

val inputsDir = new java.io.File(calibrationDir, "inputs")
profiles.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir, "user-popularity-profiles").getPath)
movieGroupsOut.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir, "movie-popularity-groups").getPath)
segments.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir, "user-segments").getPath)
trainingUsers.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir, "evaluation-users").getPath)
userHistory.select("userId", "nTrain", "historyBucket").coalesce(1).write.mode("errorifexists")
  .parquet(new java.io.File(inputsDir, "user-history").getPath)

println(s"Wrote ${allRows.size} result rows to ${csvFile.getPath}")
println(s"Evaluator inputs in ${inputsDir.getPath}")
println(s"Split: seed=$splitSeed, fraction=$trainingFraction; baseline run: $baselineRunId; popularity profile run: $profileRunId")

## 10. Cross-check with the team's UPD evaluator (recommended)

Open `upd-evaluation.ipynb`, run its definition cells, then use the paths printed above in its Section 9, pointing `recommendations` at one exported list such as `top-10-cp_l0.5`. `overallScore` should equal the `upd` value in the `all` row of `results.csv` for that setting, within `1e-9`.

## 11. Stop Spark

In [ ]:
spark.stop()
println("Spark session stopped.")